In [2]:
# Setup cell — run this first. Do not modify.

# Install the Snowflake connector if the environment does not have it yet
%pip install -q snowflake-connector-python

# Import the libraries used in this task
import requests
import pandas as pd
from decimal import Decimal
from getpass import getpass
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas


# The four cities of this task: city name and ISO country code
CITIES = [
    {"city": "Peshawar", "country_code": "PK"},
    {"city": "Quetta", "country_code": "PK"},
    {"city": "Dhaka", "country_code": "BD"},
    {"city": "Colombo", "country_code": "LK"},
]

# Air-quality variables to request (a subset of the main lab variables)
AIR_VARIABLES = "pm2_5,pm10,nitrogen_dioxide,ozone,us_aqi"

# Number of forecast days to request from the Air Quality API
FORECAST_DAYS = 3

# Name of the database used in this task
DB = "SA_AIR_TASK"


# Ask for your Snowflake details (nothing is stored in the notebook)
sf_account = input("Snowflake account identifier: ")
sf_user = input("Snowflake user name: ")
sf_password = getpass("Snowflake password: ")
sf_warehouse = input("Snowflake warehouse [SNOWFLAKE_LEARNING_WH]: ") or "SNOWFLAKE_LEARNING_WH"

# Open the connection
conn = snowflake.connector.connect(
    account=sf_account,
    user=sf_user,
    password=sf_password,
    warehouse=sf_warehouse,
)


def run_sql(sql):
    """Run ONE SQL statement in Snowflake and return the result as a DataFrame."""
    cur = conn.cursor()
    try:
        # Execute the statement
        cur.execute(sql)
        # Statements without a result set (for example CREATE) return an empty DataFrame
        if cur.description is None:
            return pd.DataFrame()
        # Fetch the rows and use the column names that Snowflake returns
        df = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        cur.close()
    # Snowflake returns decimal numbers as Decimal objects, so convert them to float
    for col in df.columns:
        if df[col].map(lambda v: isinstance(v, Decimal)).any():
            df[col] = df[col].astype(float)
    return df


# Create the database and the two schemas if they do not exist yet (this does not delete anything)
run_sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.STAGING")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.MART")

print("Setup finished. Database and schemas are ready:", DB)



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Snowflake account identifier: YSYOVZQ-QQ10633
Snowflake user name: 233670
Snowflake password: ******
Snowflake warehouse [SNOWFLAKE_LEARNING_WH]: SNOWFLAKE_LEARNING_WH
Setup finished. Database and schemas are ready: SA_AIR_TASK


In [5]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Task 9.1 - Step 1: Locations of the cities in CITIES

def get_city_location(city, country_code):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    response = requests.get(
        url,
        params={
            "name": city,
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=30
    )
    response.raise_for_status()

    results = response.json().get("results", [])

    # Keep the first result of the requested country
    match = next(
        (
            item for item in results
            if item.get("country_code", "").upper() == country_code
        ),
        None
    )

    if match is None:
        raise ValueError(f"Location not found: {city}, {country_code}")

    return {
        "CITY": match["name"],
        "COUNTRY": match.get("country"),
        "COUNTRY_CODE": match.get("country_code"),
        "LATITUDE": match["latitude"],
        "LONGITUDE": match["longitude"],
        "TIMEZONE": match.get("timezone")
    }

locations_df = pd.DataFrame([
    get_city_location(item["city"], item["country_code"])
    for item in CITIES
])

display(locations_df)

,CITY,COUNTRY,COUNTRY_CODE,LATITUDE,LONGITUDE,TIMEZONE
0,Peshawar,Pakistan,PK,34.00800,71.57849,Asia/Karachi
1,Quetta,Pakistan,PK,30.18414,67.00141,Asia/Karachi
2,Dhaka,Bangladesh,BD,23.71040,90.40744,Asia/Dhaka
3,Colombo,Sri Lanka,LK,6.93548,79.84868,Asia/Colombo


In [3]:
# Task 9.1 - Step 2: Hourly air-quality data for every city

def fetch_hourly_air(location):
    params = {
        "latitude": location["LATITUDE"],
        "longitude": location["LONGITUDE"],
        "hourly": AIR_VARIABLES,
        "forecast_days": FORECAST_DAYS,
        "timezone": "UTC"
    }

    response = requests.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        params=params,
        timeout=60
    )
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])

    # Upper-case column names as required
    df = df.rename(columns={
        "pm2_5": "PM2_5",
        "pm10": "PM10",
        "nitrogen_dioxide": "NITROGEN_DIOXIDE",
        "ozone": "OZONE",
        "us_aqi": "US_AQI"
    })

    # Text timestamp such as 2026-11-01T00:00:00Z
    df["TIME_UTC"] = pd.to_datetime(
        df["time"], utc=True
    ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    df["CITY"] = location["CITY"]
    df["COUNTRY_CODE"] = location["COUNTRY_CODE"]

    return df[[
        "PM2_5", "PM10", "NITROGEN_DIOXIDE", "OZONE",
        "US_AQI", "TIME_UTC", "CITY", "COUNTRY_CODE"
    ]]


# Combine all cities into environment_df
hourly_frames = [
    fetch_hourly_air(location)
    for location in locations_df.to_dict("records")
]

environment_df = pd.concat(hourly_frames, ignore_index=True)

print("Data collection completed.")
print("Total rows:", len(environment_df))
print("Total columns:", len(environment_df.columns))

display(environment_df.head(10))

Data collection completed.
Total rows: 288
Total columns: 8


,PM2_5,PM10,NITROGEN_DIOXIDE,OZONE,US_AQI,TIME_UTC,CITY,COUNTRY_CODE
0,46.4,51.7,36.7,8.0,107,2026-11-11T00:00:00Z,Peshawar,PK
1,46.1,51.2,36.6,4.0,107,2026-11-11T01:00:00Z,Peshawar,PK
2,54.1,59.4,36.1,0.0,108,2026-11-11T02:00:00Z,Peshawar,PK
3,64.0,69.2,33.4,6.0,108,2026-11-11T03:00:00Z,Peshawar,PK
4,43.9,51.6,26.5,35.0,110,2026-11-11T04:00:00Z,Peshawar,PK
5,34.5,43.8,17.5,75.0,111,2026-11-11T05:00:00Z,Peshawar,PK
6,26.3,36.1,10.4,107.0,113,2026-11-11T06:00:00Z,Peshawar,PK
7,25.7,36.6,5.1,127.0,114,2026-11-11T07:00:00Z,Peshawar,PK
8,26.1,37.1,1.9,139.0,115,2026-11-11T08:00:00Z,Peshawar,PK
9,25.8,36.6,5.4,139.0,117,2026-11-11T09:00:00Z,Peshawar,PK


In [4]:
# Task 9.1 - Step 3: Validate before loading

# Expected rows = number of cities x 24 hours x forecast days
expected_rows = len(CITIES) * 24 * FORECAST_DAYS
print("Expected rows:", expected_rows)
print("Actual rows:", len(environment_df))

# Records per city
print("\nRecords per city:")
display(
    environment_df.groupby(
        ["CITY", "COUNTRY_CODE"]
    ).size().reset_index(name="RECORD_COUNT")
)

# Duplicate (CITY, TIME_UTC) pairs
duplicates = environment_df.duplicated(
    subset=["CITY", "TIME_UTC"]
).sum()
print("Duplicate (CITY, TIME_UTC) pairs:", duplicates)

# Missing values per column
print("\nMissing values per column:")
display(
    environment_df.isnull()
    .sum()
    .reset_index(name="MISSING_COUNT")
    .rename(columns={"index": "COLUMN"})
)

# First and last TIME_UTC
print("First TIME_UTC:", environment_df["TIME_UTC"].min())
print("Last TIME_UTC:", environment_df["TIME_UTC"].max())

# Stop if validation fails
assert len(environment_df) == expected_rows, (
    f"Row count mismatch: expected {expected_rows}, got {len(environment_df)}"
)
assert duplicates == 0, (
    f"Found {duplicates} duplicate (CITY, TIME_UTC) pairs"
)

print("\nValidation passed.")

Expected rows: 288
Actual rows: 288

Records per city:


,CITY,COUNTRY_CODE,RECORD_COUNT
0,Colombo,LK,72
1,Dhaka,BD,72
2,Peshawar,PK,72
3,Quetta,PK,72


Duplicate (CITY, TIME_UTC) pairs: 0

Missing values per column:


,COLUMN,MISSING_COUNT
0,PM2_5,0
1,PM10,0
2,NITROGEN_DIOXIDE,0
3,OZONE,0
4,US_AQI,0
5,TIME_UTC,0
6,CITY,0
7,COUNTRY_CODE,0


First TIME_UTC: 2026-11-11T00:00:00Z
Last TIME_UTC: 2026-11-13T23:00:00Z

Validation passed.


In [5]:
# Task 9.1 - Step 4: Create staging tables, load and verify
# Create and load are in one cell, so a re-run replaces the tables and does not duplicate rows

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.STAGING.API_LOCATIONS (
    CITY VARCHAR(100),
    COUNTRY VARCHAR(100),
    COUNTRY_CODE VARCHAR(10),
    LATITUDE FLOAT,
    LONGITUDE FLOAT,
    TIMEZONE VARCHAR(100)
)
""")

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.STAGING.API_HOURLY_AIR (
    PM2_5 FLOAT,
    PM10 FLOAT,
    NITROGEN_DIOXIDE FLOAT,
    OZONE FLOAT,
    US_AQI FLOAT,
    TIME_UTC VARCHAR(30),
    CITY VARCHAR(100),
    COUNTRY_CODE VARCHAR(10)
)
""")

write_pandas(
    conn,
    locations_df[["CITY", "COUNTRY", "COUNTRY_CODE", "LATITUDE", "LONGITUDE", "TIMEZONE"]],
    table_name="API_LOCATIONS",
    database=DB,
    schema="STAGING"
)

write_pandas(
    conn,
    environment_df[["PM2_5", "PM10", "NITROGEN_DIOXIDE", "OZONE",
                    "US_AQI", "TIME_UTC", "CITY", "COUNTRY_CODE"]],
    table_name="API_HOURLY_AIR",
    database=DB,
    schema="STAGING"
)

# Verify each load
print("API_LOCATIONS rows:")
display(run_sql(f"SELECT COUNT(*) AS ROW_COUNT FROM {DB}.STAGING.API_LOCATIONS"))

print("API_HOURLY_AIR rows:")
display(run_sql(f"SELECT COUNT(*) AS ROW_COUNT FROM {DB}.STAGING.API_HOURLY_AIR"))

API_LOCATIONS rows:


,ROW_COUNT
0,4


API_HOURLY_AIR rows:


,ROW_COUNT
0,288


In [6]:
# Task 9.2 - Step 1: Create MART tables
# DIM_COUNTRY first, because DIM_LOCATION stores only its COUNTRY_KEY

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.MART.DIM_COUNTRY (
    COUNTRY_KEY INTEGER AUTOINCREMENT PRIMARY KEY,
    COUNTRY VARCHAR(100),
    COUNTRY_CODE VARCHAR(10)
)
""")

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.MART.DIM_LOCATION (
    LOCATION_KEY INTEGER AUTOINCREMENT PRIMARY KEY,
    CITY VARCHAR(100),
    COUNTRY_KEY INTEGER,
    LATITUDE FLOAT,
    LONGITUDE FLOAT,
    TIMEZONE VARCHAR(100)
)
""")

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.MART.DIM_DATETIME (
    DATETIME_KEY NUMBER(10, 0) PRIMARY KEY,
    FULL_TIMESTAMP TIMESTAMP_NTZ,
    DATE_VALUE DATE,
    DAY_NAME VARCHAR(20),
    HOUR_NUMBER INTEGER
)
""")

run_sql(f"""
CREATE OR REPLACE TABLE {DB}.MART.FACT_AIR_HOURLY (
    LOCATION_KEY INTEGER,
    DATETIME_KEY NUMBER(10, 0),
    PM2_5 FLOAT,
    PM10 FLOAT,
    NITROGEN_DIOXIDE FLOAT,
    OZONE FLOAT,
    US_AQI FLOAT
)
""")

print("MART tables created.")

MART tables created.


In [7]:
# Task 9.2 - Step 2: Load MART tables from the staging tables
# Run this cell together with the previous cell, otherwise rows are added again

# Format of the TIME_UTC text, e.g. 2026-11-01T00:00:00Z
TS_FORMAT = 'YYYY-MM-DD"T"HH24:MI:SS"Z"'

# DIM_COUNTRY: each country once
run_sql(f"""
INSERT INTO {DB}.MART.DIM_COUNTRY (COUNTRY, COUNTRY_CODE)
SELECT DISTINCT COUNTRY, COUNTRY_CODE
FROM {DB}.STAGING.API_LOCATIONS
""")

# DIM_LOCATION: take COUNTRY_KEY from DIM_COUNTRY
run_sql(f"""
INSERT INTO {DB}.MART.DIM_LOCATION (CITY, COUNTRY_KEY, LATITUDE, LONGITUDE, TIMEZONE)
SELECT
    l.CITY,
    c.COUNTRY_KEY,
    l.LATITUDE,
    l.LONGITUDE,
    l.TIMEZONE
FROM {DB}.STAGING.API_LOCATIONS AS l
JOIN {DB}.MART.DIM_COUNTRY AS c
    ON l.COUNTRY_CODE = c.COUNTRY_CODE
""")

# DIM_DATETIME: one row per distinct hour, key in the form YYYYMMDDHH
run_sql(f"""
INSERT INTO {DB}.MART.DIM_DATETIME
    (DATETIME_KEY, FULL_TIMESTAMP, DATE_VALUE, DAY_NAME, HOUR_NUMBER)
SELECT
    TO_NUMBER(TO_CHAR(ts, 'YYYYMMDDHH24')),
    ts,
    TO_DATE(ts),
    DAYNAME(ts),
    HOUR(ts)
FROM (
    SELECT DISTINCT TO_TIMESTAMP_NTZ(TIME_UTC, '{TS_FORMAT}') AS ts
    FROM {DB}.STAGING.API_HOURLY_AIR
)
""")

# FACT_AIR_HOURLY: look up both keys with joins
run_sql(f"""
INSERT INTO {DB}.MART.FACT_AIR_HOURLY
    (LOCATION_KEY, DATETIME_KEY, PM2_5, PM10, NITROGEN_DIOXIDE, OZONE, US_AQI)
SELECT
    l.LOCATION_KEY,
    d.DATETIME_KEY,
    h.PM2_5,
    h.PM10,
    h.NITROGEN_DIOXIDE,
    h.OZONE,
    h.US_AQI
FROM {DB}.STAGING.API_HOURLY_AIR AS h
JOIN {DB}.MART.DIM_LOCATION AS l
    ON h.CITY = l.CITY
JOIN {DB}.MART.DIM_DATETIME AS d
    ON TO_TIMESTAMP_NTZ(h.TIME_UTC, '{TS_FORMAT}') = d.FULL_TIMESTAMP
""")

print("MART tables loaded.")

MART tables loaded.


In [8]:
# Task 9.2 - Step 3: Integrity checks

# Row count of every table
for table in ["STAGING.API_HOURLY_AIR", "MART.DIM_COUNTRY", "MART.DIM_LOCATION",
              "MART.DIM_DATETIME", "MART.FACT_AIR_HOURLY"]:
    n = run_sql(f"SELECT COUNT(*) AS ROW_COUNT FROM {DB}.{table}").iloc[0, 0]
    print(f"{table}: {int(n)} rows")

# Fact rows must equal staged rows
staged_rows = int(run_sql(f"SELECT COUNT(*) FROM {DB}.STAGING.API_HOURLY_AIR").iloc[0, 0])
fact_rows = int(run_sql(f"SELECT COUNT(*) FROM {DB}.MART.FACT_AIR_HOURLY").iloc[0, 0])
assert fact_rows == staged_rows, f"Fact rows {fact_rows} != staged rows {staged_rows}"
print("\nFact rows equal staged rows:", fact_rows)

# Orphan check 1: fact rows without a location
orphans_location = run_sql(f"""
SELECT COUNT(*) AS ORPHANS_NO_LOCATION
FROM {DB}.MART.FACT_AIR_HOURLY AS f
LEFT JOIN {DB}.MART.DIM_LOCATION AS l
    ON f.LOCATION_KEY = l.LOCATION_KEY
WHERE l.LOCATION_KEY IS NULL
""")
display(orphans_location)

# Orphan check 2: fact rows without a date-time row
orphans_datetime = run_sql(f"""
SELECT COUNT(*) AS ORPHANS_NO_DATETIME
FROM {DB}.MART.FACT_AIR_HOURLY AS f
LEFT JOIN {DB}.MART.DIM_DATETIME AS d
    ON f.DATETIME_KEY = d.DATETIME_KEY
WHERE d.DATETIME_KEY IS NULL
""")
display(orphans_datetime)

assert orphans_location.iloc[0, 0] == 0, "Fact rows without a location"
assert orphans_datetime.iloc[0, 0] == 0, "Fact rows without a date-time"
print("No orphans found.")

STAGING.API_HOURLY_AIR: 288 rows
MART.DIM_COUNTRY: 3 rows
MART.DIM_LOCATION: 4 rows
MART.DIM_DATETIME: 72 rows
MART.FACT_AIR_HOURLY: 288 rows

Fact rows equal staged rows: 288


,ORPHANS_NO_LOCATION
0,0


,ORPHANS_NO_DATETIME
0,0


No orphans found.


In [9]:
# Task 9.2 - Step 4: Drill-down on average PM2.5

# Shared FROM/JOIN part, so it is not repeated in every level
BASE_JOIN = f"""
FROM {DB}.MART.FACT_AIR_HOURLY AS f
JOIN {DB}.MART.DIM_LOCATION AS l ON f.LOCATION_KEY = l.LOCATION_KEY
JOIN {DB}.MART.DIM_COUNTRY AS c ON l.COUNTRY_KEY = c.COUNTRY_KEY
JOIN {DB}.MART.DIM_DATETIME AS d ON f.DATETIME_KEY = d.DATETIME_KEY
"""

# Level 1: by country, highest first
level1 = run_sql(f"""
SELECT c.COUNTRY,
       ROUND(AVG(f.PM2_5), 2) AS AVG_PM2_5,
       COUNT(*) AS ROW_COUNT
{BASE_JOIN}
GROUP BY c.COUNTRY
ORDER BY AVG(f.PM2_5) DESC
""")
print("Level 1: by country")
display(level1)
top_country = level1.iloc[0]["COUNTRY"]

# Level 2: cities of the top country
level2 = run_sql(f"""
SELECT l.CITY,
       ROUND(AVG(f.PM2_5), 2) AS AVG_PM2_5,
       COUNT(*) AS ROW_COUNT
{BASE_JOIN}
WHERE c.COUNTRY = '{top_country}'
GROUP BY l.CITY
ORDER BY AVG(f.PM2_5) DESC
""")
print(f"\nLevel 2: cities of {top_country}")
display(level2)
top_city = level2.iloc[0]["CITY"]

# Level 3: dates of the top city
level3 = run_sql(f"""
SELECT d.DATE_VALUE,
       ROUND(AVG(f.PM2_5), 2) AS AVG_PM2_5,
       COUNT(*) AS ROW_COUNT
{BASE_JOIN}
WHERE c.COUNTRY = '{top_country}'
  AND l.CITY = '{top_city}'
GROUP BY d.DATE_VALUE
ORDER BY AVG(f.PM2_5) DESC
""")
print(f"\nLevel 3: dates of {top_city}")
display(level3)
top_date = level3.iloc[0]["DATE_VALUE"]

# Level 4: the 24 hours of the top date
level4 = run_sql(f"""
SELECT d.HOUR_NUMBER,
       ROUND(AVG(f.PM2_5), 2) AS AVG_PM2_5,
       COUNT(*) AS ROW_COUNT
{BASE_JOIN}
WHERE c.COUNTRY = '{top_country}'
  AND l.CITY = '{top_city}'
  AND d.DATE_VALUE = '{top_date}'
GROUP BY d.HOUR_NUMBER
ORDER BY AVG(f.PM2_5) DESC
""")
print(f"\nLevel 4: hours of {top_date}")
display(level4)
top_hour = int(level4.iloc[0]["HOUR_NUMBER"])

# Row counts of each level must add up to the group above
assert level1["ROW_COUNT"].sum() == fact_rows, "Level 1 does not add up to fact rows"
assert level2["ROW_COUNT"].sum() == level1.iloc[0]["ROW_COUNT"], "Level 2 does not add up to level 1"
assert level3["ROW_COUNT"].sum() == level2.iloc[0]["ROW_COUNT"], "Level 3 does not add up to level 2"
assert level4["ROW_COUNT"].sum() == level3.iloc[0]["ROW_COUNT"], "Level 4 does not add up to level 3"
assert len(level4) == 24, "Level 4 should show 24 hours"
print("\nAll drill-down levels add up.")

Level 1: by country


,COUNTRY,AVG_PM2_5,ROW_COUNT
0,Bangladesh,58.20,72
1,Pakistan,41.56,144
2,Sri Lanka,15.59,72



Level 2: cities of Bangladesh


,CITY,AVG_PM2_5,ROW_COUNT
0,Dhaka,58.2,72



Level 3: dates of Dhaka


,DATE_VALUE,AVG_PM2_5,ROW_COUNT
0,2026-11-13,64.82,24
1,2026-11-12,63.28,24
2,2026-11-11,46.51,24



Level 4: hours of 2026-11-13


,HOUR_NUMBER,AVG_PM2_5,ROW_COUNT
0,1,110.6,1
1,0,105.7,1
2,2,92.7,1
3,16,92.4,1
4,17,90.6,1
5,15,89.6,1
6,18,84.8,1
7,14,83.6,1
8,19,78.2,1
9,13,74.7,1



All drill-down levels add up.


In [10]:
# Task 9.2 - Step 5: Interpretation

interpretation = (
    f"The highest average PM2.5 is in {top_country}, in the city {top_city}, "
    f"on {top_date}, and the highest hour of that date is {top_hour}:00 UTC. "
    "This may not describe the real air quality because the data is a model forecast "
    "(CAMS) for only a few days and not a measurement from sensors, "
    "and a short period cannot represent the whole year."
)

print(interpretation)

The highest average PM2.5 is in Bangladesh, in the city Dhaka, on 2026-11-13, and the highest hour of that date is 1:00 UTC. This may not describe the real air quality because the data is a model forecast (CAMS) for only a few days and not a measurement from sensors, and a short period cannot represent the whole year.
